![JohnSnowLabs](https://nlp.johnsnowlabs.com/assets/images/logo.png)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/JohnSnowLabs/visual-nlp-workshop/blob/master/tutorials/Certification_Trainings_JSL/05.00.Spark_OCR_Streaming_PDF.ipynb)

If you are using the `spark-ocr` library, please use this [05.00.Spark_OCR_Streaming_PDF](https://github.com/JohnSnowLabs/visual-nlp-workshop/blob/master/tutorials/Certification_Trainings/05.00.Spark_OCR_Streaming_PDF.ipynb) notebook.

## Spark OCR Streaming

## Blogposts and videos

- [Text Detection in Spark OCR](https://medium.com/spark-nlp/text-detection-in-spark-ocr-dcd8002bdc97)

- [Table Detection & Extraction in Spark OCR](https://medium.com/spark-nlp/table-detection-extraction-in-spark-ocr-50765c6cedc9)

- [Extract Tabular Data from PDF in Spark OCR](https://medium.com/spark-nlp/extract-tabular-data-from-pdf-in-spark-ocr-b02136bc0fcb)

- [Signature Detection in Spark OCR](https://medium.com/spark-nlp/signature-detection-in-spark-ocr-32f9e6f91e3c)

- [GPU image pre-processing in Spark OCR](https://medium.com/spark-nlp/gpu-image-pre-processing-in-spark-ocr-3-1-0-6fc27560a9bb)

- [How to Setup Spark OCR on UBUNTU - Video](https://www.youtube.com/watch?v=cmt4WIcL0nI)


**More examples here**

https://github.com/JohnSnowLabs/spark-ocr-workshop

### Colab Setup

In [ ]:
# Install the johnsnowlabs library to access Spark-OCR and Spark-NLP for Healthcare, Finance, and Legal.
!pip install -q johnsnowlabs

In [ ]:
from google.colab import files
print('Please Upload your John Snow Labs License using the button below')
license_keys = files.upload()

In [ ]:
from johnsnowlabs import nlp, visual, medical

# After uploading your license run this to install all licensed Python Wheels and pre-download Jars the Spark Session JVM
nlp.install(refresh_install=True, visual=True)

In [4]:
import pyspark
import json
import os

⚠️ Important: running the next cell will **automatically restart** the Colab runtime (this is expected after installing new jars/wheels). Once it restarts, just continue running the cells below.

In [ ]:
import os
os.kill(os.getpid(), 9)

## Initialization of spark session

In [1]:
from johnsnowlabs import visual, nlp
import pandas as pd

# Automatically load license data and start a session with all jars user has access to
spark = nlp.start(visual=True)

👌 Launched cpu optimized session with with: 🚀Spark-NLP==6.4.0, 💊Spark-Healthcare==6.4.0, 🕶Spark-OCR==6.4.0, running on ⚡ PySpark==3.4.0


In [2]:
from pyspark.ml import PipelineModel
from pyspark.sql.functions import *

In [3]:
# Download a sample PDF from the repo
!wget -q -O /content/noised.pdf https://raw.githubusercontent.com/JohnSnowLabs/visual-nlp-workshop/master/tutorials/Certification_Trainings_JSL/data/pdfs/noised.pdf

# fill path to folder with PDF's here
dataset_path = "/content/*.pdf"

In [4]:
# read one file for infer schema
pdfs_df = spark.read.format("binaryFile").load(dataset_path).limit(1)

## Define OCR pipeline

In [17]:
# Transform binary to image
pdf_to_image = visual.PdfToImage()
pdf_to_image.setOutputCol("image")

# Run OCR for each region
ocr = visual.ImageToText()
ocr.setInputCol("image")
ocr.setOutputCol("text")
ocr.setConfidenceThreshold(60)

# OCR pipeline
pipeline = PipelineModel(stages=[
    pdf_to_image,
    ocr
])

## Define streaming pipeline and start it
Note: each start erase previous results

In [18]:
# count of files in one microbatch
maxFilesPerTrigger = 4

# read files as stream
pdf_stream_df = spark.readStream \
.format("binaryFile") \
.schema(pdfs_df.schema) \
.option("maxFilesPerTrigger", maxFilesPerTrigger) \
.load(dataset_path)

# process files using OCR pipeline
result = pipeline.transform(pdf_stream_df).withColumn("timestamp", current_timestamp())

# store results to memory table
query = result.writeStream \
 .format('memory') \
 .queryName('result') \
 .start()

In [19]:
import time
time.sleep(10)

# get progress of streamig job
query.lastProgress

{'id': '94d18b5b-c3be-4d88-a2ce-761da45884ba',
 'runId': '057b7ca7-9520-438c-b4bb-8dc36cb58cab',
 'name': 'result',
 'timestamp': '2026-07-08T12:23:14.995Z',
 'batchId': 0,
 'numInputRows': 1,
 'inputRowsPerSecond': 0.0,
 'processedRowsPerSecond': 0.20462451401677922,
 'durationMs': {'addBatch': 4416,
  'commitOffsets': 56,
  'getBatch': 34,
  'latestOffset': 118,
  'queryPlanning': 153,
  'triggerExecution': 4887,
  'walCommit': 101},
 'stateOperators': [],
 'sources': [{'description': 'FileStreamSource[file:/content/*.pdf]',
   'startOffset': None,
   'endOffset': {'logOffset': 0},
   'latestOffset': None,
   'numInputRows': 1,
   'inputRowsPerSecond': 0.0,
   'processedRowsPerSecond': 0.20462451401677922}],
 'sink': {'description': 'MemorySink', 'numOutputRows': 1}}

In [20]:
time.sleep(10)

# need to run for stop steraming job
query.stop()

## Show results from 'result' table
Remember to upload some file to the /content folder in colab.

In [21]:
# count of processed records (number of processed pages in results)
spark.table("result").count()

1

In [22]:
# show results
spark.table("result").select("timestamp","pagenum", "path", "text").show(10)

+--------------------+-------+--------------------+--------------------+
|           timestamp|pagenum|                path|                text|
+--------------------+-------+--------------------+--------------------+
|2026-07-08 12:23:...|      0|file:/content/noi...| \n\n \n\n \n\nSa...|
+--------------------+-------+--------------------+--------------------+



## Run streaming job for storing results to disk

In [28]:
# format: could also be parquet, or csv
# path: route to a file system location
query = result.select("text").writeStream \
 .format('text') \
 .option("path", "results/") \
 .option("checkpointLocation", "checkpointDir") \
 .start()

In [29]:
time.sleep(10)

# get progress of streamig job
query.lastProgress

{'id': '1f0a2d8e-ab02-4f6a-b988-d741cc8e099a',
 'runId': '14f08c77-803d-45d1-9460-80066c2626e5',
 'name': None,
 'timestamp': '2026-07-08T12:25:37.939Z',
 'batchId': 0,
 'numInputRows': 1,
 'inputRowsPerSecond': 0.0,
 'processedRowsPerSecond': 0.21496130696474633,
 'durationMs': {'addBatch': 4226,
  'commitOffsets': 71,
  'getBatch': 11,
  'latestOffset': 106,
  'queryPlanning': 125,
  'triggerExecution': 4651,
  'walCommit': 97},
 'stateOperators': [],
 'sources': [{'description': 'FileStreamSource[file:/content/*.pdf]',
   'startOffset': None,
   'endOffset': {'logOffset': 0},
   'latestOffset': None,
   'numInputRows': 1,
   'inputRowsPerSecond': 0.0,
   'processedRowsPerSecond': 0.21496130696474633}],
 'sink': {'description': 'FileSink[results/]', 'numOutputRows': -1}}

In [30]:
time.sleep(10)

# need to run for stop steraming job
query.stop()

## Read results from disk

In [31]:
results = spark.read.format("text").load("results/*.txt")
results.show(50, truncate=False)

+--------------------------------------------------------------------+
|value                                                               |
+--------------------------------------------------------------------+
|                                                                    |
|                                                                    |
|                                                                    |
|                                                                    |
|                                                                    |
|                                                                    |
|Sample specifications written by                                    |
|                                                                    |
|                                                                    |
|                                                                    |
|~ , BLEND CASING RECASING                                           |
|= OLD

In [32]:
results.sample(.1).show(truncate=False)

+--------------------------+
|value                     |
+--------------------------+
|                          |
|MENTHOL FLAVOR            |
|Shipping ----------- Tot .|
+--------------------------+



## Clean results and checkpoint folders

In [33]:
%%bash
rm -r -f results
rm -r -f checkpointDir